Database Connection

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Check Unique Values

In [7]:
# ║  BLOCK 1 — Μοναδικές τιμές ανά categorical column, ανά dataset             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Τρέξε αυτό πρώτο σε ένα cell του notebook
import pandas as pd
from sqlalchemy import text
# ── Categorical columns ──
# Κοινά πεδία (υπάρχουν και στα δύο datasets)
common_categoricals = ['body_part', 'play_pattern', 'last_action', 'outcome', 'h_a']
# StatsBomb-only πεδία
sb_only_categoricals = ['shot_type', 'technique', 'position']
with engine.connect() as conn:
    print("=" * 80)
    print("ΚΟΙΝΑ CATEGORICAL ΠΕΔΙΑ — Σύγκριση τιμών")
    print("=" * 80)
    for col in common_categoricals:
        # StatsBomb
        sb_vals = pd.read_sql(
            text(f"SELECT DISTINCT {col} FROM statsbomb_shots ORDER BY {col}"),
            conn
        )[col].dropna().tolist()
        # Understat
        us_vals = pd.read_sql(
            text(f"SELECT DISTINCT {col} FROM understat_shots ORDER BY {col}"),
            conn
        )[col].dropna().tolist()
        print(f"\n{'─' * 80}")
        print(f"📊  {col.upper()}")
        print(f"{'─' * 80}")
        print(f"  {'StatsBomb':<40} {'Understat':<40}")
        print(f"  {'─' * 38}   {'─' * 38}")
        max_rows = max(len(sb_vals), len(us_vals))
        for i in range(max_rows):
            sb_val = sb_vals[i] if i < len(sb_vals) else ""
            us_val = us_vals[i] if i < len(us_vals) else ""
            print(f"  {str(sb_val):<40} {str(us_val):<40}")
        # Εμφάνιση πλήθους
        print(f"\n  Πλήθος: SB={len(sb_vals)}, US={len(us_vals)}")
    print(f"\n\n{'=' * 50}")
    print("STATSBOMB-ONLY CATEGORICAL ΠΕΔΙΑ")
    print("=" * 50)
    for col in sb_only_categoricals:
        sb_vals = pd.read_sql(
            text(f"SELECT DISTINCT {col} FROM statsbomb_shots ORDER BY {col}"),
            conn
        )[col].dropna().tolist()

        print(f"\n{'─' * 50}")
        print(f"📊  {col.upper()} (StatsBomb only)")
        print(f"{'─' * 50}")
        for v in sb_vals:
            print(f"  • {v}")
        print(f"  Πλήθος: {len(sb_vals)}")

ΚΟΙΝΑ CATEGORICAL ΠΕΔΙΑ — Σύγκριση τιμών

────────────────────────────────────────────────────────────────────────────────
📊  BODY_PART
────────────────────────────────────────────────────────────────────────────────
  StatsBomb                                Understat                               
  ──────────────────────────────────────   ──────────────────────────────────────
  Head                                     Head                                    
  Left Foot                                LeftFoot                                
  Other                                    OtherBodyPart                           
  Right Foot                               RightFoot                               

  Πλήθος: SB=4, US=4

────────────────────────────────────────────────────────────────────────────────
📊  PLAY_PATTERN
────────────────────────────────────────────────────────────────────────────────
  StatsBomb                                Understat                            

Matching Values

In [8]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BLOCK 2 — Αντιστοίχιση πεδίων Understat ↔ StatsBomb (1 προς 1)           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Τρέξε αυτό σε δεύτερο cell

print("\n")
print("=" * 100)
print("ΠΛΗΡΗΣ ΑΝΤΙΣΤΟΙΧΙΣΗ ΠΕΔΙΩΝ: understat_shots ↔ statsbomb_shots")
print("=" * 100)
print()

# (understat_column, statsbomb_column, σχόλιο)
field_mapping = [
    # ── ΑΝΑΓΝΩΡΙΣΤΙΚΑ ──
    ("shot_id (INTEGER)",          "shot_id (VARCHAR/UUID)",       "⚠️ Διαφορετικός τύπος: INT vs UUID"),
    ("match_id",                   "match_id",                     "✅ Ίδιο"),
    ("uid_understat",              "player_id",                    "⚠️ Διαφορετικό ID system — χρειάζεται mapping table"),

    # ── ΧΡΟΝΟΣ & CONTEXT ──
    ("minute",                     "minute",                       "✅ Ίδιο"),
    ("—",                          "second",                       "❌ Δεν υπάρχει στο Understat"),
    ("—",                          "period",                       "❌ Δεν υπάρχει στο Understat"),
    ("season (SMALLINT, π.χ. 2020)", "season (VARCHAR, π.χ. '2020/2021')", "⚠️ Διαφορετικό format"),
    ("league",                     "competition",                  "⚠️ Ίδια πληροφορία, πιθανώς διαφορετικά ονόματα"),
    ("h_a",                        "h_a",                          "✅ Ίδιο ('h'/'a')"),
    ("—",                          "position",                     "❌ Θέση παίκτη — δεν υπάρχει στο Understat"),
    ("—",                          "team_id",                      "❌ Δεν υπάρχει στο Understat"),
    ("—",                          "team_name",                    "❌ Δεν υπάρχει στο Understat"),

    # ── ΓΕΩΜΕΤΡΙΑ ──
    ("x_loc (0–1)",                "x_loc (0–120 yards)",          "⚠️ ΔΙΑΦΟΡΕΤΙΚΗ ΚΛΙΜΑΚΑ — χρειάζεται normalization"),
    ("y_loc (0–1)",                "y_loc (0–80 yards)",           "⚠️ ΔΙΑΦΟΡΕΤΙΚΗ ΚΛΙΜΑΚΑ — χρειάζεται normalization"),
    ("distance_to_goal (100×100)", "distance_to_goal (120×80)",    "⚠️ ΑΣΥΓΚΡΙΤΑ — πρέπει recompute σε κοινή κλίμακα"),
    ("angle_to_goal (100×100)",    "angle_to_goal (120×80)",       "⚠️ ΑΣΥΓΚΡΙΤΑ — πρέπει recompute σε κοινή κλίμακα"),
    ("—",                          "end_x",                        "❌ Τελική θέση μπάλας — δεν υπάρχει στο Understat"),
    ("—",                          "end_y",                        "❌"),
    ("—",                          "end_z",                        "❌"),

    # ── ΔΗΜΙΟΥΡΓΙΑ ΦΑΣΗΣ ──
    ("play_pattern",               "play_pattern",                 "⚠️ Ίδια πληροφορία — ΔΙΑΦΟΡΕΤΙΚΕΣ ΤΙΜΕΣ (βλ. Block 1)"),
    ("last_action",                "last_action",                  "⚠️ Ίδια πληροφορία — ΔΙΑΦΟΡΕΤΙΚΕΣ ΤΙΜΕΣ (βλ. Block 1)"),
    ("player_assisted",            "player_assisted",              "✅ Ίδιο (όνομα, nullable)"),
    ("uid_assisted",               "player_assisted_id",           "⚠️ Διαφορετικό ID system"),
    ("—",                          "key_pass_id",                  "❌ UUID assist event — δεν υπάρχει στο Understat"),

    # ── ΜΗΧΑΝΙΚΗ ΣΟΥΤ ──
    ("body_part",                  "body_part",                    "⚠️ Ίδια πληροφορία — ΔΙΑΦΟΡΕΤΙΚΕΣ ΤΙΜΕΣ (βλ. Block 1)"),
    ("—",                          "shot_type",                    "❌ StatsBomb only"),
    ("—",                          "technique",                    "❌ StatsBomb only"),
    ("—",                          "first_time",                   "❌ StatsBomb only (boolean)"),

    # ── SITUATIONAL FLAGS ──
    ("—",                          "under_pressure",               "❌ StatsBomb only (boolean)"),
    ("—",                          "one_on_one",                   "❌ StatsBomb only (boolean)"),
    ("—",                          "open_goal",                    "❌ StatsBomb only (boolean)"),
    ("—",                          "aerial_won",                   "❌ StatsBomb only (boolean)"),
    ("—",                          "follows_dribble",              "❌ StatsBomb only (boolean)"),
    ("—",                          "redirect",                     "❌ StatsBomb only (boolean)"),
    ("—",                          "deflected",                    "❌ StatsBomb only (boolean)"),

    # ── ΑΠΟΤΕΛΕΣΜΑ ──
    ("outcome",                    "outcome",                      "⚠️ Ίδια πληροφορία — ΔΙΑΦΟΡΕΤΙΚΕΣ ΤΙΜΕΣ (βλ. Block 1)"),
    ("is_goal",                    "is_goal",                      "✅ Ίδιο (0/1)"),
    ("understat_xg",               "statsbomb_xg",                 "✅ Ίδιος ρόλος — provider xG (benchmark)"),
]

# Εκτύπωση ως πίνακα
print(f"  {'#':<4} {'UNDERSTAT':<35} {'STATSBOMB':<35} {'ΣΧΟΛΙΟ'}")
print(f"  {'─'*4} {'─'*35} {'─'*35} {'─'*45}")

for i, (us, sb, note) in enumerate(field_mapping, 1):
    print(f"  {i:<4} {us:<35} {sb:<35} {note}")

print(f"\n{'─' * 100}")
print("ΣΥΝΟΨΗ:")
print(f"  ✅ Ταυτόσημα πεδία:              {sum(1 for _,_,n in field_mapping if n.startswith('✅'))}")
print(f"  ⚠️ Χρειάζονται harmonization:    {sum(1 for _,_,n in field_mapping if n.startswith('⚠'))}")
print(f"  ❌ StatsBomb-only (NaN για US):   {sum(1 for _,_,n in field_mapping if n.startswith('❌'))}")



ΠΛΗΡΗΣ ΑΝΤΙΣΤΟΙΧΙΣΗ ΠΕΔΙΩΝ: understat_shots ↔ statsbomb_shots

  #    UNDERSTAT                           STATSBOMB                           ΣΧΟΛΙΟ
  ──── ─────────────────────────────────── ─────────────────────────────────── ─────────────────────────────────────────────
  1    shot_id (INTEGER)                   shot_id (VARCHAR/UUID)              ⚠️ Διαφορετικός τύπος: INT vs UUID
  2    match_id                            match_id                            ✅ Ίδιο
  3    uid_understat                       player_id                           ⚠️ Διαφορετικό ID system — χρειάζεται mapping table
  4    minute                              minute                              ✅ Ίδιο
  5    —                                   second                              ❌ Δεν υπάρχει στο Understat
  6    —                                   period                              ❌ Δεν υπάρχει στο Understat
  7    season (SMALLINT, π.χ. 2020)        season (VARCHAR, π.χ. '2020/2021')  ⚠️ Διαφορετικό

Unique Values

In [9]:
def get_unique(col, table):
    with engine.connect() as conn:
        query = f"SELECT DISTINCT {col} FROM {table} ORDER BY {col}"
        return pd.read_sql(text(query), conn)[col].dropna().tolist()

print("=" * 60)
print("1. SEASON")
print("Understat:", get_unique('season', 'understat_shots'))
print("StatsBomb:", get_unique('season', 'statsbomb_shots'))
print("-" * 60)

print("2. LEAGUE / COMPETITION")
print("Understat (league):     ", get_unique('league', 'understat_shots'))
print("StatsBomb (competition):", get_unique('competition', 'statsbomb_shots'))
print("-" * 60)

print("3. PLAY_PATTERN")
print("Understat:", get_unique('play_pattern', 'understat_shots'))
print("StatsBomb:", get_unique('play_pattern', 'statsbomb_shots'))
print("-" * 60)

print("4. LAST_ACTION")
print("Understat:", get_unique('last_action', 'understat_shots'))
print("StatsBomb:", get_unique('last_action', 'statsbomb_shots'))
print("-" * 60)

print("5. BODY_PART")
print("Understat:", get_unique('body_part', 'understat_shots'))
print("StatsBomb:", get_unique('body_part', 'statsbomb_shots'))
print("-" * 60)

print("6. OUTCOME")
print("Understat:", get_unique('outcome', 'understat_shots'))
print("StatsBomb:", get_unique('outcome', 'statsbomb_shots'))
print("=" * 60)

1. SEASON
Understat: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
StatsBomb: ['2015/2016', '2016/2017', '2017/2018', '2018', '2018/2019', '2019/2020', '2020', '2020/2021', '2021/2022', '2022', '2022/2023', '2023', '2023/2024', '2024']
------------------------------------------------------------
2. LEAGUE / COMPETITION
Understat (league):      ['Bundesliga', 'EPL', 'La_Liga', 'Ligue_1', 'RFPL', 'Serie_A']
StatsBomb (competition): ['1. Bundesliga', 'African Cup of Nations', 'Champions League', 'Copa America', 'FIFA World Cup', 'Indian Super league', 'La Liga', 'Ligue 1', 'Major League Soccer', 'Premier League', 'Serie A', 'UEFA Euro']
------------------------------------------------------------
3. PLAY_PATTERN
Understat: ['DirectFreekick', 'FromCorner', 'OpenPlay', 'Penalty', 'SetPiece']
StatsBomb: ['From Corner', 'From Counter', 'From Free Kick', 'From Goal Kick', 'From Keeper', 'From Kick Off', 'From Throw In', 'Other', 'Regular Play']
----------------------

Normalize Values + Distance to Meters

In [31]:
import numpy as np

print("Φόρτωση δεδομένων από τη βάση...")
df_understat_raw = pd.read_sql("SELECT * FROM understat_shots", engine)
df_statsbomb_raw = pd.read_sql("SELECT * FROM statsbomb_shots", engine)
print(f"✔️ Φορτώθηκαν {len(df_understat_raw)} σουτ από Understat και {len(df_statsbomb_raw)} από StatsBomb.\n")

# 2. ΣΥΝΑΡΤΗΣΗ ΕΝΑΡΜΟΝΙΣΗΣ (HARMONIZATION)
def normalize_datasets(df_us, df_sb):
    df_understat = df_us.copy()
    df_statsbomb = df_sb.copy()

    # --- SEASON FORMATTING ---
    if df_statsbomb['season'].dtype == 'O':
        df_statsbomb['season'] = df_statsbomb['season'].astype(str).str.split('/').str[0].astype(int)

    # --- LEAGUE / COMPETITION ---
    league_map_us = {
        'EPL': 'Premier League', 'La_Liga': 'La Liga', 'Ligue_1': 'Ligue 1',
        'Serie_A': 'Serie A', 'RFPL': 'Russian Premier League', 'Bundesliga': 'Bundesliga'
    }
    df_understat['league'] = df_understat['league'].map(league_map_us).fillna(df_understat['league'])
    df_statsbomb['competition'] = df_statsbomb['competition'].replace({'1. Bundesliga': 'Bundesliga'})

    # --- PLAY PATTERN ---
    pp_us = {'OpenPlay': 'Open Play', 'FromCorner': 'Corner', 'DirectFreekick': 'Free Kick',
             'SetPiece': 'Set Piece', 'Penalty': 'Penalty'}
    pp_sb = {'Regular Play': 'Open Play', 'From Counter': 'Counter', 'From Corner': 'Corner',
             'From Free Kick': 'Free Kick', 'From Throw In': 'Set Piece', 'From Keeper': 'Set Piece',
             'From Goal Kick': 'Set Piece', 'From Kick Off': 'Set Piece', 'Other': 'Other'}

    df_understat['play_pattern'] = df_understat['play_pattern'].map(pp_us).fillna('Other')
    df_statsbomb['play_pattern'] = df_statsbomb['play_pattern'].map(pp_sb).fillna('Other')

    # --- BODY PART ---
    bp_us = {'RightFoot': 'Right Foot', 'LeftFoot': 'Left Foot', 'Head': 'Head', 'OtherBodyPart': 'Other'}
    df_understat['body_part'] = df_understat['body_part'].map(bp_us).fillna('Other')

    # --- OUTCOME ---
    out_us = {'Goal': 'Goal', 'SavedShot': 'Saved', 'BlockedShot': 'Blocked',
              'MissedShots': 'Off Target', 'ShotOnPost': 'Post', 'OwnGoal': 'Own Goal'}
    out_sb = {'Goal': 'Goal', 'Saved': 'Saved', 'Saved Off Target': 'Saved', 'Saved to Post': 'Saved',
              'Blocked': 'Blocked', 'Off T': 'Off Target', 'Wayward': 'Off Target', 'Post': 'Post'}

    df_understat['outcome'] = df_understat['outcome'].map(out_us).fillna('Other')
    df_statsbomb['outcome'] = df_statsbomb['outcome'].map(out_sb).fillna('Other')

    # --- LAST ACTION ---
    def map_last_action(a):
        a = str(a).lower()
        if a in ['pass', 'throughball', 'chipped', 'layoff', 'headpass']: return 'Pass'
        if 'cross' in a: return 'Cross'
        if a in ['carry', 'dribble', 'takeon', 'goodskill']: return 'Dribble/Carry'
        if a in ['ball receipt*', 'balltouch', 'miscontrol', 'aerial']: return 'Receipt/Control'
        if a in ['clearance', 'ball recovery', 'ballrecovery', 'interception', 'tackle',
                 'challenge', 'duel', 'block', 'blockedpass']: return 'Recovery/Duel'
        if a == 'rebound': return 'Rebound'
        if a in ['error', 'dispossessed']: return 'Error'
        if 'foul' in a or 'offside' in a: return 'Foul'
        if 'keeper' in a or a in ['save', 'smother', 'punch', 'claim']: return 'Goal Keeper'
        return 'Other'

    df_understat['last_action'] = df_understat['last_action'].apply(map_last_action)
    df_statsbomb['last_action'] = df_statsbomb['last_action'].apply(map_last_action)

    # --- GEOMETRY NORMALIZATION (105x68 meters) ---
    df_understat['x_meters'] = df_understat['x_loc'] * 105.0
    df_understat['y_meters'] = df_understat['y_loc'] * 68.0

    df_statsbomb['x_meters'] = (df_statsbomb['x_loc'] / 120.0) * 105.0
    df_statsbomb['y_meters'] = (df_statsbomb['y_loc'] / 80.0) * 68.0

    def calc_geometry(df):
        df['distance_to_goal_meters'] = np.sqrt((105.0 - df['x_meters'])**2 + (34.0 - df['y_meters'])**2)
        df['angle_to_goal_rad'] = np.abs(
            np.arctan2(37.66 - df['y_meters'], 105.0 - df['x_meters']) -
            np.arctan2(30.34 - df['y_meters'], 105.0 - df['x_meters'])
        )
        return df

    df_understat = calc_geometry(df_understat)
    df_statsbomb = calc_geometry(df_statsbomb)

    return df_understat, df_statsbomb

# ==============================================================================
# 3. ΕΚΤΕΛΕΣΗ ΚΑΙ ΕΚΤΥΠΩΣΗ ΑΠΟΤΕΛΕΣΜΑΤΩΝ
# ==============================================================================
print("Εφαρμογή εναρμόνισης (Harmonization)...")
df_us, df_sb = normalize_datasets(df_understat_raw, df_statsbomb_raw)
print("✔️ Η εναρμόνιση ολοκληρώθηκε επιτυχώς!\n")

print("="*60)
print("ΔΕΙΓΜΑ ΕΝΑΡΜΟΝΙΣΜΕΝΩΝ ΚΑΤΗΓΟΡΙΩΝ (StatsBomb)")
print("="*60)
print("OUTCOME (Unique Values):")
print(df_sb['outcome'].value_counts())
print("\nBODY PART (Unique Values):")
print(df_sb['body_part'].value_counts())

print("\n" + "="*60)
print("ΕΠΙΒΕΒΑΙΩΣΗ ΓΕΩΜΕΤΡΙΑΣ (ΜΕΣΟΣ ΟΡΟΣ)")
print("="*60)
print(f"Understat - Μέση Απόσταση (m): {df_us['distance_to_goal_meters'].mean():.2f}")
print(f"StatsBomb - Μέση Απόσταση (m): {df_sb['distance_to_goal_meters'].mean():.2f}")

Φόρτωση δεδομένων από τη βάση...
✔️ Φορτώθηκαν 615225 σουτ από Understat και 56749 από StatsBomb.

Εφαρμογή εναρμόνισης (Harmonization)...
✔️ Η εναρμόνιση ολοκληρώθηκε επιτυχώς!

ΔΕΙΓΜΑ ΕΝΑΡΜΟΝΙΣΜΕΝΩΝ ΚΑΤΗΓΟΡΙΩΝ (StatsBomb)
OUTCOME (Unique Values):
outcome
Off Target    21723
Blocked       14237
Saved         13539
Goal           6118
Post           1132
Name: count, dtype: int64

BODY PART (Unique Values):
body_part
Right Foot    29228
Left Foot     18338
Head           9018
Other           165
Name: count, dtype: int64

ΕΠΙΒΕΒΑΙΩΣΗ ΓΕΩΜΕΤΡΙΑΣ (ΜΕΣΟΣ ΟΡΟΣ)
Understat - Μέση Απόσταση (m): 18.42
StatsBomb - Μέση Απόσταση (m): 16.77


Small Normalization tricks

In [32]:
# Δημιουργία Boolean "is_assisted", true εαν υπάρχει assist, false εάν δεν υπάρχει
df_us['is_assisted'] = df_us['player_assisted'].notna().astype(int)
df_sb['is_assisted'] = df_sb['player_assisted'].notna().astype(int)

# Column renaming
df_understat = df_us.rename(columns={
    'league': 'competition',
    'uid_understat': 'player_id',
    'uid_assisted': 'player_assisted_id',
    'understat_xg': 'provider_xg'
})

df_statsbomb = df_sb.rename(columns={
    'statsbomb_xg': 'provider_xg'
})

# Remove default distances
cols_to_drop = ['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal']
df_understat = df_understat.drop(columns=cols_to_drop)
df_statsbomb = df_statsbomb.drop(columns=cols_to_drop)

# Rename columns with meters
rename_dict = {
    'x_meters': 'x_loc',
    'y_meters': 'y_loc',
    'distance_to_goal_meters': 'distance_to_goal',
    'angle_to_goal_rad': 'angle_to_goal'
}
df_understat = df_understat.rename(columns=rename_dict)
df_statsbomb = df_statsbomb.rename(columns=rename_dict)

Remove own goals

In [33]:
# StatsBomb
print("Αυτογκόλ στο StatsBomb πριν τον καθαρισμό:", (df_statsbomb['outcome'] == 'Own Goal').sum())
df_statsbomb = df_statsbomb[df_statsbomb['outcome'] != 'Own Goal']

# Understat
print("Αυτογκόλ στο Understat πριν τον καθαρισμό:", (df_understat['outcome'] == 'Own Goal').sum())
df_understat = df_understat[df_understat['outcome'] != 'Own Goal']

Αυτογκόλ στο StatsBomb πριν τον καθαρισμό: 0
Αυτογκόλ στο Understat πριν τον καθαρισμό: 1961


Delete penalties from main dataframes and put them into new one for future exploration

In [34]:
# Στο StatsBomb το βρίσκουμε από το shot_type
df_statsbomb_penalties = df_statsbomb[df_statsbomb['shot_type'] == 'Penalty'].copy()

# Στο Understat το βρίσκουμε από το play_pattern
df_understat_penalties = df_understat[df_understat['play_pattern'] == 'Penalty'].copy()
print(f"{len(df_statsbomb_penalties)} πέναλτι στο StatsBomb")
print(f"{len(df_understat_penalties)} πέναλτι στο Understat")

# Delete penalties from main dataframes
df_statsbomb = df_statsbomb[df_statsbomb['shot_type'] != 'Penalty']
df_understat = df_understat[df_understat['play_pattern'] != 'Penalty']

892 πέναλτι στο StatsBomb
7818 πέναλτι στο Understat


Insert to DB (4 new tables)

In [38]:
print(f"Αποθήκευση StatsBomb ({len(df_statsbomb)}) + ({len(df_statsbomb_penalties)}) στη βάση...")
df_statsbomb.to_sql('statsbomb_shots_normalized', engine, if_exists='replace', index=False)
df_statsbomb_penalties.to_sql('statsbomb_shots_penalties_normalized', engine, if_exists='replace', index=False)

print("Αποθήκευση Understat ({len(df_understat)}) + ({len(df_understat_penalties)}) στη βάση...")
df_understat.to_sql('understat_shots_normalized', engine, if_exists='replace', index=False)
df_understat_penalties.to_sql('understat_shots_penalties_normalized', engine, if_exists='replace', index=False)

print("✅ Όλα τα δεδομένα αποθηκεύτηκαν επιτυχώς")

Αποθήκευση StatsBomb (55857) + (892) στη βάση...
Αποθήκευση Understat ({len(df_understat)}) + ({len(df_understat_penalties)}) στη βάση...
✅ Όλα τα δεδομένα αποθηκεύτηκαν επιτυχώς και ομοιόμορφα!
